# Spectral Flow and Many-Body Charge Pump

A one-dimensional flux scan probes the low-energy spectrum and transverse
polarization of an interacting torus. This notebook uses the bosonic Haldane
model on 2×3 unit cells and the public observable APIs.

Flux arguments are **in units of 2π**. One cycle is `0 ≤ θx ≤ 1`;
`collect(0.0:1/16:1.0)` contains 17 points including both endpoints.
Every momentum sector is solved at every point. The initial lowest sectors
need not remain lowest as flux changes.

## Gauge-covariant translations and spectral flow

The boundary hopping for a move from site $i$ to $j$ carries its winding phase
$e^{2\pi i\theta_x w_{ij}}$. For a translation $g$, the matching site phase is

$$\eta_g(x)=\exp\!\left[2\pi i\sum_\mu\theta_\mu
\frac{x_\mu-(gx)_\mu}{L_\mu}\right].$$

With $G_\theta=\exp(2\pi i\sum_j\theta\cdot x_j/L\,n_j)$, the operation is
$T^\theta=G_\theta^{-1}T^0G_\theta$. Its eigenvalues use the same irrep labels
as at zero twist. Under a full flux quantum, spectra in these labelled sectors
can permute. These are labels of the basis, not a promise that one sector
contains the same ground state throughout the cycle.

The scan builds the flux-dependent symmetry group and orbit catalog at each
point. Plot a constant reference $E_n(\theta)-E_0(0)$ to see continuous energy
branches. Subtracting $E_0(\theta)$ instead plots excitation gaps and pins the
instantaneous ground branch to zero.

## Global low-energy manifold and projected polarization

At each flux, merge the computed eigenpairs from **all** sectors, sort by
energy, and choose the lowest $m=\texttt{manifold_size}$ states. The solver
retains at least $m+1$ states per sector to measure $E_{m+1}-E_m$ globally.
Several selected states may belong to the same momentum sector.

In the transverse direction, define

$$U_y=\exp\!\left(\frac{2\pi i}{L_y}\sum_j y_j n_j\right),\qquad
M_{ab}(\theta)=\langle\psi_a(\theta)|U_y|\psi_b(\theta)\rangle.$$

Coordinates include sublattice offsets by default. $U_y$ is unitary, but
its projection $M$ need not be unitary: its eigenvalues can have magnitude
less than one. The routine requires a positive manifold gap and nonsingular
$M$; otherwise its polarization is not reported.

The phases of the eigenvalues of $M$ are matched and unwrapped between
adjacent fluxes. In units of particle charge,

$$P_b(\theta)=\frac{\operatorname{unwrap}\arg\lambda_b(\theta)}{2\pi},
\qquad Q_b(\theta)=P_b(\theta)-P_b(0).$$

These polarization eigenbranches generally mix energy states and momentum
sectors. They must not be labelled as individual sector eigenstates. A coarse
grid can miss phase winding or gap minima; refine it when branch matching is
ambiguous.

In [1]:
using RealSpace_ExactDiagonalization, CairoMakie, Test, Logging
repo = isfile("Project.toml") ? pwd() : dirname(pwd())
cache = joinpath(repo, "checkpoints", "notebooks", "charge_pump")
figures = joinpath(repo, "figures", "notebooks", "charge_pump")
mkpath(figures)
model = build_zero_flux_bosonic_fci_second_quantized_model(; sample_size=[2, 3])
fluxes = collect(0.0:1/16:1.0)
filling = 1 // 4   # 3 hard-core bosons on 12 graph vertices

# Keep verbose ED/checkpoint logs in the cache directory.
function with_ed_log(f, name)
    mkpath(cache)
    open(joinpath(cache, name * ".log"), "w") do io
        redirect_stdout(io) do
            redirect_stderr(io) do
                with_logger(SimpleLogger(io)) do
                    f()
                end
            end
        end
    end
end
nothing

In [2]:
flow = with_ed_log("flow") do
    flux_spectrum_flow(model, :all; filling_fraction=filling, nev=4,
        twisted_phases_over_2π_list=fluxes, checkpoint_dir=cache)
end
reference = minimum(flow.energies[1, :, :])
fig = Figure(size=(800, 500))
ax = Axis(fig[1, 1]; xlabel="inserted x flux / 2π", ylabel="E − E₀(0)")
for (index, label) in enumerate(flow.sector_labels), level in 1:flow.nev
    lines!(ax, fluxes, flow.energies[:, index, level] .- reference;
        color=Cycled(index), alpha=level == 1 ? 1.0 : 0.3,
        label=level == 1 ? string(label) : nothing)
end
axislegend(ax; nbanks=2)
save(joinpath(figures, "spectrum_flow.svg"), fig)
@test flow.global_energies[1, :] ≈ flow.global_energies[end, :] atol=1e-9
println("Scanned ", length(flow.sector_labels), " sectors at ", length(fluxes), " flux points.")

Scanned 6 sectors at 17 flux points.


In [3]:
pump = with_ed_log("pump") do
    flux_charge_pump(model, :all; filling_fraction=filling, manifold_size=2,
        flux_direction=1, polarization_direction=2,
        twisted_phases_over_2π_list=fluxes, checkpoint_dir=cache,
        fig_path=joinpath(figures, "charge_pump.svg"))
end
@test pump.pumped_charges ≈ [0.5, 0.5] atol=1e-8
@test pump.min_gap > 0
@test minimum(pump.min_position_singular_values) > 1e-10
println("Branch charges: ", pump.pumped_charges)
println("Total charge: ", sum(pump.pumped_charges))
println("Minimum gap: ", pump.min_gap)
println("Selected states at the final flux: ", pump.selected_states[end])

Branch charges: [0.5000000000000001, 0.49999999999999967]
Total charge: 0.9999999999999998
Minimum gap: 0.36256914952301766
Selected states at the final flux: NamedTuple[(sector = (1, 0), level = 1), (sector = (0, 0), level = 1)]


## Comparison with the many-body Chern number

`many_body_chern_number` computes the **integer Chern number of the entire
isolated manifold** using determinant overlap links over the two-dimensional
flux torus. It also selects the global lowest $m$ states at every grid point
and rejects singular links or a closed gap. The non-Abelian lattice method is
described by [Fukui, Hatsugai and Suzuki (2005)](https://arxiv.org/abs/cond-mat/0503172).

The finite-size Resta pump is a related diagnostic. It is not defined by
rounding its endpoint to an integer, and individual finite-size branches need
not carry identical fractions. For the small bosonic example here the pump
branches give 1/2 each, while the two-state manifold has Chern number 1.
The fractional Hall response of an isolated, equally weighted topological
multiplet is associated with its Chern number divided by its dimension.
Check size and grid convergence when making that identification.

In [4]:
chern = with_ed_log("chern") do
    many_body_chern_number(model, :all; filling_fraction=filling,
        manifold_size=2, flux_grid_size=(5, 5), checkpoint_dir=joinpath(cache, "chern"),
        fig_path=joinpath(figures, "chern_number.svg"))
end
@test chern.chern_number ≈ 1 atol=1e-8
println("Manifold Chern number: ", chern.chern_number)
println("Minimum gap: ", chern.min_gap, "; minimum link determinant: ", chern.min_link_det)

Manifold Chern number: 1.0
Minimum gap: 0.29557407915615386; minimum link determinant: 0.6583441079398972


## Checkpoints and HPC

Use `checkpoint_dir` for observable scans. Checkpoints contain a solver revision
and parameter fingerprint; incompatible data are rejected. Sector labels supplied
to flow restrict display only. For pump and Chern calculations, use `:all` with
an explicit `manifold_size`.

The [phase-exploration HPC guide](../phase_exploration/hpc/README.md) configures
17 shared flux points for flow and pump, writes `spectrum_flow.csv` and
`charge_pump.csv`, and records the selected states, gap, and projected-position
singular values. It performs no two-dimensional Chern grid by default.

See [design.ipynb](design.ipynb) for symmetry projection and
[observables.ipynb](observables.ipynb) for correlation observables.